In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

# -----------------------------
# CONFIG (EDIT ONLY THIS PART)
# -----------------------------
PROJECT_DIR = r"C:\Users\admin\Desktop\Brain_Stroke_Project"
MODELS_DIR  = os.path.join(PROJECT_DIR, "models")

RESNET_MODEL_PATH = os.path.join(MODELS_DIR, "resnet50_best.h5")
EFF_MODEL_PATH    = os.path.join(MODELS_DIR, "efficientnetb0_best.h5")

IMG_SIZE  = 224
THRESHOLD = 0.6       # your tuned threshold
W_RESNET  = 0.5
W_EFF     = 0.5

# -----------------------------
# LOAD MODELS
# -----------------------------
def load_models():
    if not os.path.exists(RESNET_MODEL_PATH):
        raise FileNotFoundError(f"ResNet model not found: {RESNET_MODEL_PATH}")
    if not os.path.exists(EFF_MODEL_PATH):
        raise FileNotFoundError(f"EfficientNet model not found: {EFF_MODEL_PATH}")

    print("✅ Loading models...")
    resnet_model = load_model(RESNET_MODEL_PATH)
    eff_model = load_model(EFF_MODEL_PATH)
    print("✅ Models loaded successfully.\n")
    return resnet_model, eff_model

# -----------------------------
# PREPROCESS IMAGE FOR A MODEL
# -----------------------------
import cv2
import numpy as np

IMG_SIZE = 224

def ct_preprocess_single(img_path):
    img = cv2.imread(img_path)                # BGR
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)

    gray = cv2.GaussianBlur(gray, (3,3), 0)

    # convert back to 3 channels (models expect 3 channels)
    x = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32)

    x = np.expand_dims(x, axis=0)  # (1,224,224,3)

    # If your training used rescale 1/255:
    x = x / 255.0

    return x


# -----------------------------
# ENSEMBLE PREDICTION
# -----------------------------
def classify_image(img_path, resnet_model, eff_model):
    x_resnet = ct_preprocess_single(img_path)
    x_eff    = ct_preprocess_single(img_path)
    
    p_resnet = float(resnet_model.predict(x_resnet, verbose=0).ravel()[0])
    p_eff    = float(eff_model.predict(x_eff, verbose=0).ravel()[0])


    # Ensemble probability
    p_final = W_RESNET * p_resnet + W_EFF * p_eff

    # Final label
    label = "Stroke" if p_final >= THRESHOLD else "Normal"

    return label, p_final, p_resnet, p_eff

# -----------------------------
# MAIN
# -----------------------------
def main():
    resnet_model, eff_model = load_models()

    print("📌 Paste CT image full path when asked (example):")
    print(r"C:\Users\admin\Desktop\Brain_Stroke_Project\dataset\test\Stroke\image1.jpg")
    print()

    img_path = input("Enter CT image path: ").strip().strip('"')

    if not os.path.exists(img_path):
        print("\n❌ Image not found. Check the path and try again.")
        return

    label, p_final, p_resnet, p_eff = classify_image(img_path, resnet_model, eff_model)

    print("\n==============================")
    print("✅ FINAL CLASSIFICATION OUTPUT")
    print("==============================")
    print(f"Image Path  : {img_path}")
    print(f"ResNet Prob : {p_resnet:.3f}")
    print(f"EffNet Prob : {p_eff:.3f}")
    print(f"Ensemble Prob (final): {p_final:.3f}")
    print(f"Threshold  : {THRESHOLD}")
    print(f"Prediction : {label}")
    print("==============================\n")

if __name__ == "__main__":
    main()


✅ Loading models...
✅ Models loaded successfully.

📌 Paste CT image full path when asked (example):
C:\Users\admin\Desktop\Brain_Stroke_Project\dataset\test\Stroke\image1.jpg


✅ FINAL CLASSIFICATION OUTPUT
Image Path  : C:\Users\admin\Desktop\Brain_Stroke_Project\dataset\Test\Stroke\67 (27).jpg
ResNet Prob : 0.877
EffNet Prob : 0.505
Ensemble Prob (final): 0.691
Threshold  : 0.6
Prediction : Stroke

